In [40]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd
from numpy import nan
from functions import functions
import json

from urllib.parse import quote_plus
from sqlalchemy import create_engine
engine = create_engine('postgresql://zbrothers:%s@localhost:5434/zbrothers' % quote_plus('zbrothers@2026'))

In [2]:
API_TOKEN = os.environ['OLIST_API_TOKEN']

today = datetime.date.today()
# today = datetime.datetime(2026, 4, 15)
today_str = today.strftime('%d/%m/%Y')

situacao_dict = {
    'aguardando_separacao': 1,
    'em_separacao': 4,
    'separada': 2,
    'embalada': 3
}

In [3]:
def getAllProductsPicking(situacao_dict: dict, today_str: str) -> pd.DataFrame:
    df_list = []
    for situacao, situacao_id in situacao_dict.items():
        print(f'Getting data for situacao: {situacao} (id: {situacao_id})')
        df = functions.getInfosPesquisar(
            url='https://api.tiny.com.br/api2/separacao.pesquisa.php',
            params={
                'token': API_TOKEN,
                'dataInicial': today_str,
                'situacao': situacao_id
            }
        )
        df['situacao'] = situacao
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)

In [4]:
df_all_products_picking = getAllProductsPicking(situacao_dict, today_str)

Getting data for situacao: aguardando_separacao (id: 1)
{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacoes': [{'id': '1075559156', 'idOrigem': '1075555738', 'objOrigem': 'notafiscal', 'situacao': '1', 'dataCriacao': '28/04/2026', 'dataSeparacao': None, 'dataCheckout': None, 'destinatario': 'Carlos Alberto Dos Santos Junior', 'idContato': '1031485118', 'numero': '188765', 'dataEmissao': '28/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000016170316478', 'idOrigemVinc': '1075551978', 'objOrigemVinc': 'venda', 'situacaoVenda': '6'}, {'id': '1075559106', 'idOrigem': '1075555864', 'objOrigem': 'notafiscal', 'situacao': '1', 'dataCriacao': '28/04/2026', 'dataSeparacao': None, 'dataCheckout': None, 'destinatario': 'Thiago Oliveira Mendes', 'idContato': '1031484299', 'numero': '188787', 'dataEmissao': '28/04/2026', 'idFormaEnvio': '675234044', 'numeroPedidoEcommerce': '2000016161940106', 'idOrigemVinc': '1075543391', 'objOrigemVinc': 'venda', 'situacaoVen

In [5]:
df_all_products_picking_detail = functions.getInfosObter('https://api.tiny.com.br/api2/separacao.obter.php', 
                                                         {'token': API_TOKEN, 'formato': 'json', 'idSeparacao': None},
                                                         df_all_products_picking['id'].tolist(),
                                                         'idSeparacao')

{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacao': {'id': '1075559156', 'idOrigem': '1075555738', 'objOrigem': 'notafiscal', 'situacao': '1', 'situacaoCheckout': '1', 'dataCriacao': '28/04/2026', 'itens': [{'idProduto': '1069373421', 'descricao': 'LPL-750 LIXADEIRA PAREDE LED LYNUS 750W 127V', 'codigo': '000114707', 'quantidade': '1.0000', 'unidade': 'UN', 'localizacao': '', 'infoAdicional': ''}], 'qtdVolumes': '0', 'numero': '188765', 'dataEmissao': '28/04/2026', 'numeroPedidoEcommerce': '2000016170316478', 'idFormaEnvio': '675234044', 'formaEnvio': 'Mercado Envios', 'idContato': '1031485118', 'destinatario': 'Carlos Alberto Dos Santos Junior', 'situacaoOrigem': None}}}
{'retorno': {'status_processamento': '3', 'status': 'OK', 'separacao': {'id': '1075559106', 'idOrigem': '1075555864', 'objOrigem': 'notafiscal', 'situacao': '1', 'situacaoCheckout': '1', 'dataCriacao': '28/04/2026', 'itens': [{'idProduto': '1074237446', 'descricao': 'MACHO MANUAL AR  8X1,00MB JG 2PC

In [18]:
df_all_products_picking

,id,idOrigem,objOrigem,situacao,dataCriacao,dataSeparacao,dataCheckout,destinatario,idContato,numero,dataEmissao,idFormaEnvio,numeroPedidoEcommerce,idOrigemVinc,objOrigemVinc,situacaoVenda
0,1075559156,1075555738,notafiscal,aguardando_separacao,28/04/2026,None,None,Carlos Alberto Dos Santos Junior,1031485118,188765,28/04/2026,675234044,2000016170316478,1075551978,venda,6
1,1075559106,1075555864,notafiscal,aguardando_separacao,28/04/2026,None,None,Thiago Oliveira Mendes,1031484299,188787,28/04/2026,675234044,2000016161940106,1075543391,venda,6
2,1075559056,1075555884,notafiscal,aguardando_separacao,28/04/2026,None,None,Marco Aurelio Magnago Alves,1031485233,188791,28/04/2026,675234044,2000012707737095,1075552809,venda,6
3,1075559052,1075556002,notafiscal,aguardando_separacao,28/04/2026,None,None,Washington Junior Matos Silva,1031362137,188809,28/04/2026,675234044,2000016171941130,1075552990,venda,6
4,1075559035,1075556077,notafiscal,aguardando_separacao,28/04/2026,None,None,Salvador Peluso Basile Neto,1031484453,188828,28/04/2026,675234044,2000016163675858,1075545064,venda,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,1075558086,1075558063,notafiscal,embalada,28/04/2026,28/04/2026,28/04/2026,Lindiney Pereira Dias,1031485469,189132,28/04/2026,737931940,26042812JVJFDC,1075553773,venda,6
403,1075558085,1075558072,notafiscal,embalada,28/04/2026,28/04/2026,28/04/2026,Sayonara Helman Palitot,1031485475,189133,28/04/2026,737931940,26042812RQF03C,1075553846,venda,6
404,1075558100,1075558078,notafiscal,embalada,28/04/2026,28/04/2026,28/04/2026,Wdson Fraga,1031485508,189134,28/04/2026,737931940,26042814AMU8XE,1075554223,venda,6
405,1075558098,1075558087,notafiscal,embalada,28/04/2026,28/04/2026,28/04/2026,Carla Xavier da Costa,1031485510,189135,28/04/2026,737931940,26042814F7RV1Y,1075554237,venda,6


In [21]:
df_all_products_picking_merged = df_all_products_picking_detail.merge(df_all_products_picking, on='id', how='left', validate='1:1')#.drop(columns=['id_y']).rename(columns={'id_x': 'id'})

In [23]:
df_all_products_picking_merged.columns

Index(['id', 'idOrigem_x', 'objOrigem_x', 'situacao_x', 'situacaoCheckout',
       'dataCriacao_x', 'itens', 'qtdVolumes', 'numero_x', 'dataEmissao_x',
       'numeroPedidoEcommerce_x', 'idFormaEnvio_x', 'formaEnvio',
       'idContato_x', 'destinatario_x', 'situacaoOrigem', 'dataSeparacao_x',
       'idUsuarioEmbalador', 'dataCheckout_x', 'idOrigem_y', 'objOrigem_y',
       'situacao_y', 'dataCriacao_y', 'dataSeparacao_y', 'dataCheckout_y',
       'destinatario_y', 'idContato_y', 'numero_y', 'dataEmissao_y',
       'idFormaEnvio_y', 'numeroPedidoEcommerce_y', 'idOrigemVinc',
       'objOrigemVinc', 'situacaoVenda'],
      dtype='str')

In [24]:
df_all_products_picking_columns = df_all_products_picking_merged[['id', 'idOrigem_x', 'objOrigem_x', 'situacao_x', 'situacaoCheckout',
       'dataCriacao_x', 'itens', 'qtdVolumes', 'numero_x', 'dataEmissao_x',
       'numeroPedidoEcommerce_x', 'idFormaEnvio_x', 'formaEnvio',
       'idContato_x', 'destinatario_x', 'situacaoOrigem', 'dataSeparacao_x',
       'idUsuarioEmbalador', 'dataCheckout_x', 'idOrigemVinc',
       'objOrigemVinc', 'situacaoVenda']]

In [68]:
df_all_products_picking_rename = df_all_products_picking_columns.rename(columns={x: x.replace('_x', '') for x in df_all_products_picking_columns.columns})\
    .rename(columns={'idUsuarioEmbalador': 'picking_operator_id'})

df_all_products_picking_rename["itens"] = df_all_products_picking_rename["itens"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if x is not None else None
)

df_all_products_picking_rename[['dataCriacao', 'dataEmissao', 'dataSeparacao', 'dataCheckout']] = df_all_products_picking_rename[['dataCriacao', 'dataEmissao', 'dataSeparacao', 'dataCheckout']].apply(lambda x: pd.to_datetime(x, format='%d/%m/%Y'))

In [73]:
embaladores_unicos = df_all_products_picking_rename[['picking_operator_id']].drop_duplicates().dropna()

In [74]:
embaladores_unicos.rename(columns={'picking_operator_id': 'id'})

,id
132,1240202884
154,879165082
307,1084497214


In [75]:
embaladores_unicos.rename(columns={'picking_operator_id': 'id'}).to_sql("picking_operators", engine, if_exists='append', index=False)

3

In [82]:
pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet] [fastparquet]
Note: you may need to restart the kernel to use updated packages.


In [83]:
df_all_products_picking_rename.to_parquet('df_all_products_picking_rename.parquet', index=False)

In [84]:
df = pd.read_parquet('df_all_products_picking_rename.parquet')

In [76]:
df_all_products_picking_rename.to_sql('product_picking', engine, if_exists='append', index=False)

407